In [ ]:
# eval_champions.ipynb
# STAGE 1 ONLY: the full downstream battery (TAG k-fold / identity /
# text-variant / sentiment / recommendation) on the BEST checkpoint of every
# DONE full-n (n=2000) combo, parallelised one worker per GPU (claim files make
# it safe across GPUs AND across VMs -- run this notebook on several pods to go
# faster). grid_metrics.json is republished after EVERY finished combo
# (streaming), so downstream consumers always see the current state.
#
# Champion selection lives in eval_curve_prepare.ipynb (reads grid_metrics.json).
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG_DIR = "/workspace/stable_query_latent_logs"
os.makedirs(LOG_DIR, exist_ok=True)
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
SWEEP_YAML = "VICReg_review/sweep/sweep.yaml"
CLAIM_TTL = 7200      # seconds before a stale per-combo eval claim is reclaimable

# Stable machine identity for claims (survives reboots where the hostname /
# container id changes): derived from this bundle's folder name Pod_N -> VMN.
# Run from the repo's Pod/ instead of a bundle -> falls back to the hostname;
# set it explicitly there if you want reboot-proof identity.
import re as _re
_m = _re.fullmatch(r'Pod_(\d+)', os.path.basename(os.getcwd()))
WORKER_ID = f'VM{_m.group(1)}' if _m else __import__('socket').gethostname()


print('repo:', REPO)
print('out :', OUT_DIR)
print('id  :', WORKER_ID)


In [ ]:
# FORCE-sync to origin/main (pod repo is a mirror; untracked files untouched).
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD


In [ ]:
# Warmup ONCE before fanning out: builds/loads the shared raw/description/text
# caches so N workers don't all embed the same texts concurrently.
!python -u VICReg_review/eval_battery_worker.py --warmup-only \
  --out-dir {OUT_DIR} --sweep-yaml {SWEEP_YAML} --worker-id {WORKER_ID} \
  --logout-address {LOG_DIR}/eval_battery_warmup.log


In [ ]:
# One battery worker per GPU. Claims coordinate across GPUs and VMs; each
# finished combo triggers a streaming republish of grid_metrics.json.
import os, subprocess, sys, time
from pathlib import Path

def _gpus():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=5).stdout
        return [l.strip() for l in out.splitlines() if l.strip()] or ['0']
    except Exception:
        return ['0']

gpus = _gpus()
procs = []
for g in gpus:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    log = f'{LOG_DIR}/eval_battery_gpu{g}.log'
    p = subprocess.Popen(
        [sys.executable, '-u', 'VICReg_review/eval_battery_worker.py',
         '--out-dir', OUT_DIR, '--sweep-yaml', SWEEP_YAML,
         '--claim-ttl', str(CLAIM_TTL), '--worker-id', WORKER_ID,
         '--logout-address', log],
        env=env, cwd=REPO)
    procs.append((g, p, log))
    print(f'worker gpu{g} pid={p.pid} -> {log}')

for g, p, log in procs:
    rc = p.wait()
    tail = ''
    try:
        tail = Path(log).read_text(encoding='utf-8', errors='replace').splitlines()[-1]
    except Exception:
        pass
    print(f'worker gpu{g} exit={rc}  last: {tail}')


In [ ]:
# Current battery state from the streaming grid_metrics.json.
import json
from pathlib import Path

mp = Path(REPO) / OUT_DIR / 'grid_metrics.json'
if not mp.exists():
    print('grid_metrics.json not written yet')
else:
    m = json.loads(mp.read_text(encoding='utf-8'))
    print(f"grid_metrics.json @ {m['created_at']} (commit {m.get('git_commit', '')[:9]})")
    print(f"  evaluated : {m['pool_size']}")
    print(f"  missing   : {len(m['missing_reports'])} (not done or not evaluated yet)")
    print('next: run eval_curve_prepare.ipynb to select champions from this file.')
